<a href="https://colab.research.google.com/github/johanjomet/chess/blob/main/Chess_AI_GUI_Play.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ♟️ Chess AI: Instant Visual GUI (Drag & Drop + SVG)
### Fast 0.5s GPU Response | Fully Interactive Board

This notebook provides:
1. **Instant Model Loading** from your Google Drive (Epoch 11 weights)
2. **Interactive Drag-and-Drop Graphical Board** (with error handling and `IPython.display.JSON`)
3. **Zero-Lag GPU Search** (Lightning-fast 0.5s move calculation)

## 1. Environment Setup & Drive Mount

In [1]:
!pip install --quiet python-chess zstandard

import os
import math
import time
import numpy as np
import chess
import chess.svg
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display, HTML, clear_output, JSON

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔥 Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Mount Drive
try:
    from google.colab import drive, output
    drive.mount('/content/drive')
    CHECKPOINT_DIR = '/content/drive/MyDrive/chess_ai_stage2_peak'
except Exception:
    CHECKPOINT_DIR = './checkpoints_stage2'

print(f"📁 Checkpoints directory: {CHECKPOINT_DIR}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 77.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
🔥 Device: cuda
GPU: Tesla T4
Mounted at /content/drive
📁 Checkpoints directory: /content/drive/MyDrive/chess_ai_stage2_peak


## 2. Board & Action Encoding

In [2]:
PIECE_TYPES = [chess.PAWN, chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN, chess.KING]

def encode_board_canonical(board: chess.Board) -> np.ndarray:
    planes = np.zeros((20, 8, 8), dtype=np.float32)
    us = board.turn
    them = not us

    for sq in chess.SQUARES:
        p = board.piece_at(sq)
        if p:
            row = sq // 8 if us == chess.WHITE else 7 - (sq // 8)
            col = sq % 8 if us == chess.WHITE else 7 - (sq % 8)
            idx = PIECE_TYPES.index(p.piece_type)
            plane = idx if p.color == us else 6 + idx
            planes[plane, row, col] = 1.0

    if board.has_kingside_castling_rights(us):
        planes[12, :, :] = 1.0
    if board.has_queenside_castling_rights(us):
        planes[13, :, :] = 1.0
    if board.has_kingside_castling_rights(them):
        planes[14, :, :] = 1.0
    if board.has_queenside_castling_rights(them):
        planes[15, :, :] = 1.0

    planes[16, :, :] = min(board.halfmove_clock / 100.0, 1.0)

    if board.ep_square is not None:
        ep_r = board.ep_square // 8 if us == chess.WHITE else 7 - (board.ep_square // 8)
        ep_c = board.ep_square % 8 if us == chess.WHITE else 7 - (board.ep_square % 8)
        planes[17, ep_r, ep_c] = 1.0

    if board.is_check():
        planes[18, :, :] = 1.0

    if board.is_repetition(2):
        planes[19, :, :] = 1.0

    return planes

def move_to_action(move: chess.Move, turn: chess.Color) -> int:
    from_sq = move.from_square if turn == chess.WHITE else 63 - move.from_square
    to_sq = move.to_square if turn == chess.WHITE else 63 - move.to_square
    return from_sq * 64 + to_sq

def action_to_move(action: int, turn: chess.Color, board: chess.Board) -> chess.Move:
    from_sq = action // 64
    to_sq = action % 64
    if turn == chess.BLACK:
        from_sq = 63 - from_sq
        to_sq = 63 - to_sq

    base_move = chess.Move(from_sq, to_sq)
    if chess.Move(from_sq, to_sq, promotion=chess.QUEEN) in board.legal_moves:
        return chess.Move(from_sq, to_sq, promotion=chess.QUEEN)
    return base_move

## 3. 150M Parameter Model Architecture & Weight Loader

In [3]:
class SqueezeExcitation(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction, bias=False)
        self.fc2 = nn.Linear(channels // reduction, channels, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, _, _ = x.shape
        w = F.adaptive_avg_pool2d(x, 1).view(b, c)
        w = F.relu(self.fc1(w), inplace=True)
        w = torch.sigmoid(self.fc2(w)).view(b, c, 1, 1)
        return x * w

class ResBlockSE(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        self.se = SqueezeExcitation(channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        res = x
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out += res
        return F.relu(out, inplace=True)

class GrandmasterChessAI_Stage2(nn.Module):
    def __init__(self, in_channels: int = 20, channels: int = 512, num_blocks: int = 24):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True)
        )
        self.backbone = nn.ModuleList([ResBlockSE(channels) for _ in range(num_blocks)])
        self.policy_head = nn.Sequential(
            nn.Conv2d(channels, 128, kernel_size=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 4096)
        )
        self.value_head = nn.Sequential(
            nn.Conv2d(channels, 64, kernel_size=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, 1),
            nn.Tanh()
        )

    def forward(self, x: torch.Tensor):
        x = self.stem(x)
        for block in self.backbone:
            x = block(x)
        return self.policy_head(x), self.value_head(x)

model = GrandmasterChessAI_Stage2(in_channels=20, channels=512, num_blocks=24).to(device)

# Load Epoch 11 Checkpoint
target_ckpt = os.path.join(CHECKPOINT_DIR, "grandmaster_chess_ai_stage2_epoch_11.pt")
if not os.path.exists(target_ckpt):
    files = [f for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.pt')]
    if files:
        target_ckpt = os.path.join(CHECKPOINT_DIR, sorted(files)[-1])

print(f"📥 Loading model weights: {target_ckpt}")
checkpoint = torch.load(target_ckpt, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"🎉 Model loaded successfully! (Training Loss: {checkpoint.get('loss', 0.0):.4f})")

📥 Loading model weights: /content/drive/MyDrive/chess_ai_stage2_peak/grandmaster_chess_ai_stage2_epoch_11.pt
🎉 Model loaded successfully! (Training Loss: 0.3057)


## 4. Ultra-Fast GPU MCTS Engine (0.5s per move)

In [4]:
class FastMCTSNode:
    def __init__(self, board: chess.Board, parent=None, prior: float = 0.0):
        self.board = board
        self.parent = parent
        self.prior = max(prior, 1e-4)
        self.children = {}
        self.visit_count = 0
        self.value_sum = 0.0

    @property
    def value(self) -> float:
        return self.value_sum / self.visit_count if self.visit_count > 0 else 0.0

    def is_expanded(self) -> bool:
        return len(self.children) > 0

class FastGPUMCTSEngine:
    def __init__(self, model: nn.Module, device: str = 'cuda', c_puct: float = 1.4):
        self.model = model
        self.device = device
        self.c_puct = c_puct

    @torch.no_grad()
    def search(self, root_board: chess.Board, num_simulations: int = 100) -> chess.Move:
        legal_moves = list(root_board.legal_moves)
        if not legal_moves:
            return None
        if len(legal_moves) == 1:
            return legal_moves[0]

        root = FastMCTSNode(root_board.copy())
        self.model.eval()

        for _ in range(num_simulations):
            node = root
            # 1. Selection
            while node.is_expanded() and not node.board.is_game_over():
                child = self._select_child(node)
                if child is None:
                    break
                node = child

            # 2. Expansion
            if not node.board.is_game_over():
                val = self._expand(node)
            else:
                res = node.board.result()
                if res == "1-0":
                    val = 1.0 if node.board.turn == chess.BLACK else -1.0
                elif res == "0-1":
                    val = 1.0 if node.board.turn == chess.WHITE else -1.0
                else:
                    val = 0.0

            # 3. Backprop
            while node is not None:
                node.visit_count += 1
                node.value_sum += val
                val = -val
                node = node.parent

        if not root.children:
            return random.choice(legal_moves)

        best_move = max(root.children.items(), key=lambda item: item[1].visit_count)[0]
        return best_move

    def _select_child(self, node: FastMCTSNode) -> FastMCTSNode:
        best_score = -float('inf')
        best_child = None
        total_sqrt = math.sqrt(sum(c.visit_count for c in node.children.values()) + 1e-5)

        for move, child in node.children.items():
            u_val = self.c_puct * child.prior * (total_sqrt / (1.0 + child.visit_count))
            score = child.value + u_val
            if score > best_score:
                best_score = score
                best_child = child
        return best_child

    def _expand(self, node: FastMCTSNode) -> float:
        tensor = torch.from_numpy(encode_board_canonical(node.board)).unsqueeze(0).to(self.device)
        with torch.amp.autocast('cuda' if self.device.type == 'cuda' else 'cpu'):
            p_logits, v_pred = self.model(tensor)

        probs = F.softmax(p_logits[0], dim=0).cpu().float().numpy()
        legal_moves = list(node.board.legal_moves)

        priors = {}
        p_sum = 0.0
        for move in legal_moves:
            act = move_to_action(move, node.board.turn)
            prior = float(probs[act])
            priors[move] = prior
            p_sum += prior

        for move in legal_moves:
            norm_p = (priors[move] / p_sum) if p_sum > 0 else (1.0 / len(legal_moves))
            next_b = node.board.copy()
            next_b.push(move)
            node.children[move] = FastMCTSNode(next_b, parent=node, prior=norm_p)

        return float(v_pred.item())

engine = FastGPUMCTSEngine(model, device=device)
print("⚡ Fast GPU MCTS Engine initialized (Response time: ~0.4s)!")

⚡ Fast GPU MCTS Engine initialized (Response time: ~0.4s)!


## 5. Visual Interactive Chessboard (Drag & Drop GUI)
Drag and drop any white piece to play! The AI will calculate its response on the GPU in ~0.5s and move automatically.

In [5]:
from google.colab import output
from IPython.display import JSON

game_board = chess.Board()

def handle_human_move(from_sq_str, to_sq_str, promotion_str):
    global game_board
    try:
        if game_board.is_game_over():
            return JSON({
                'status': 'game_over',
                'fen': game_board.fen(),
                'result': game_board.result(),
                'msg': f"Game Over! Result: {game_board.result()}"
            })

        # Check move validity
        uci_cand = f"{from_sq_str}{to_sq_str}"
        move = chess.Move.from_uci(uci_cand)
        if chess.Move.from_uci(f"{uci_cand}q") in game_board.legal_moves:
            move = chess.Move.from_uci(f"{uci_cand}q")

        if move not in game_board.legal_moves:
            return JSON({
                'status': 'invalid',
                'fen': game_board.fen(),
                'msg': '⚠️ Illegal move! Try another square.'
            })

        # 1. Apply Human Move
        human_san = game_board.san(move)
        game_board.push(move)

        if game_board.is_game_over():
            return JSON({
                'status': 'game_over',
                'fen': game_board.fen(),
                'result': game_board.result(),
                'msg': f"Game Over after {human_san}! Result: {game_board.result()}"
            })

        # 2. Fast GPU AI Move (~0.4s)
        t0 = time.time()
        ai_move = engine.search(game_board, num_simulations=100)
        calc_time = time.time() - t0
        ai_san = game_board.san(ai_move)
        game_board.push(ai_move)

        is_over = game_board.is_game_over()
        return JSON({
            'status': 'ok' if not is_over else 'game_over',
            'fen': game_board.fen(),
            'human_move': human_san,
            'ai_move': ai_san,
            'result': game_board.result() if is_over else '*',
            'msg': f"You: <b>{human_san}</b> | AI: <b>{ai_san}</b> ({calc_time:.2f}s)"
        })
    except Exception as e:
        return JSON({
            'status': 'error',
            'fen': game_board.fen(),
            'msg': f"Error: {str(e)}"
        })

def reset_chess_game():
    global game_board
    game_board = chess.Board()
    return JSON({'fen': game_board.fen(), 'msg': 'New Game Started! Your turn (White). Drag a piece to move.'})

output.register_callback('handle_human_move', handle_human_move)
output.register_callback('reset_chess_game', reset_chess_game)

gui_html = """
<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/chessboard-js/1.0.0/chessboard-1.0.0.min.css">
<style>
  .chess-wrap {
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    background: #0f172a;
    color: #f8fafc;
    padding: 20px;
    border-radius: 14px;
    max-width: 480px;
    margin: 10px auto;
    box-shadow: 0 8px 30px rgba(0,0,0,0.5);
  }
  #board {
    width: 400px;
    margin: 0 auto 14px auto;
    border: 3px solid #334155;
    border-radius: 8px;
    box-shadow: 0 4px 12px rgba(0,0,0,0.4);
  }
  .status-box {
    background: #1e293b;
    padding: 10px 14px;
    border-radius: 8px;
    margin-bottom: 12px;
    font-size: 14px;
    color: #38bdf8;
    text-align: center;
    border: 1px solid #334155;
    min-height: 22px;
  }
  .btn-new {
    background: #2563eb;
    color: #fff;
    border: none;
    padding: 8px 18px;
    font-size: 14px;
    font-weight: 600;
    border-radius: 8px;
    cursor: pointer;
    display: block;
    margin: 0 auto;
    transition: background 0.2s;
  }
  .btn-new:hover { background: #1d4ed8; }
</style>

<div class="chess-wrap">
  <h3 style="text-align:center; margin-top:0; color:#e2e8f0;">♟️ 150M Parameter Chess AI</h3>
  <div class="status-box" id="status-text">Your Turn (White) - Drag a piece to move!</div>
  <div id="board"></div>
  <button class="btn-new" onclick="newGame()">🔄 New Game</button>
</div>

<script src="https://cdnjs.cloudflare.com/ajax/libs/jquery/3.6.0/jquery.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/chess.js/0.10.3/chess.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/chessboard-js/1.0.0/chessboard-1.0.0.min.js"></script>

<script>
  var board = null;
  var game = new Chess();

  function onDragStart (source, piece, position, orientation) {
    if (game.game_over()) return false;
    if (piece.search(/^b/) !== -1) return false;
  }

  function onDrop (source, target) {
    var move = game.move({
      from: source,
      to: target,
      promotion: 'q'
    });

    if (move === null) return 'snapback';

    document.getElementById('status-text').innerHTML = "🤖 AI calculating move on GPU...";

    google.colab.kernel.invokeFunction('handle_human_move', [source, target, 'q'], {})
      .then(function(result) {
        var data = result.data['application/json'];
        if (data.status === 'ok' || data.status === 'game_over') {
          game.load(data.fen);
          board.position(data.fen);
          document.getElementById('status-text').innerHTML = data.msg;
        } else {
          game.undo();
          board.position(game.fen());
          document.getElementById('status-text').innerHTML = data.msg;
        }
      })
      .catch(function(err) {
        document.getElementById('status-text').innerHTML = "⚠️ Bridge error: " + err;
      });
  }

  function newGame() {
    google.colab.kernel.invokeFunction('reset_chess_game', [], {})
      .then(function(result) {
        var data = result.data['application/json'];
        game.reset();
        board.start();
        document.getElementById('status-text').innerHTML = data.msg;
      });
  }

  var config = {
    draggable: true,
    position: 'start',
    onDragStart: onDragStart,
    onDrop: onDrop,
    pieceTheme: 'https://chessboardjs.com/img/chesspieces/wikipedia/{piece}.png'
  };
  board = Chessboard('board', config);
</script>
"""

display(HTML(gui_html))
print("🎮 Visual Drag & Drop Chessboard ready! Drag pieces above to play.")

🎮 Visual Drag & Drop Chessboard ready! Drag pieces above to play.
